### Kütüphaneler ve Veri Yükleme
Bu adımda gerekli kütüphaneleri import edip ham verileri yüklüyoruz.
- items.csv: 962K ürün kataloğu
- terms.csv: 50K arama terimi
- training_pairs.csv: 250K pozitif çift 
- submission_pairs.csv: 3.3M tahmin edilecek çift


In [2]:
import pandas as pd
import numpy as np
import re
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

PATH = "C:/Users/acely/OneDrive/Masaüstü/trendyol_final/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Cihaz: {device}")

print("Veriler yükleniyor...")
items = pd.read_csv(PATH + "items.csv")
terms = pd.read_csv(PATH + "terms.csv")
train = pd.read_csv(PATH + "training_pairs.csv")
sub   = pd.read_csv(PATH + "submission_pairs.csv")

print("✅ Veriler yüklendi")
print(f"items: {len(items):,} | terms: {len(terms):,} | train: {len(train):,} | sub: {len(sub):,}")

Cihaz: cuda
Veriler yükleniyor...
✅ Veriler yüklendi
items: 962,873 | terms: 50,153 | train: 250,000 | sub: 3,359,679


## Veri Temizleme ve item_text Oluşturma

**Amaç:** Her ürün için modelin okuyabileceği tek ve temiz bir metin oluşturmak.

**item_text nedir:**
Model 6 ayrı kolona (title, category, brand...) bakamaz. 
Hepsini tek metinde birleştirip modele veriyoruz.

**Neler dahil edildi ve neden:**
- **title** → en önemli bilgi, ürünü tanımlıyor
- **category** → "sneaker" query'si gelince category'de "spor ayakkabı" geçiyor, eşleşme sağlıyor
- **brand** → kullanıcılar marka ile arama yapıyor (%10.7 query marka içeriyor)
- **önemli attributes** → renk, materyal, boyut gibi özellikler query ile eşleşebilir

**Neler çıkarıldı ve neden:**
- **gender/age_group** → %61 oranında "unknown", modele gürültü ekler
- **gereksiz attributes** → menşei, yıkama talimatı gibi bilgiler hiç aranmıyor

**Metin temizleme neden:**
- Küçük harf → "Adidas" ve "adidas" aynı kelime, model karıştırmasın
- Noktalama → !, %, - gibi karakterler anlamsız, modeli yanıltır

In [3]:
def clean_text(text):
    if not text or str(text).strip() == "":
        return ""
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_important_attributes(attr_text):
    if not attr_text or str(attr_text).strip() == "" or str(attr_text) == "unknown":
        return ""
    important_keys = ["renk", "materyal", "boyut", "ebat", "beden",
                      "model", "tip", "tür", "stil", "desen", "kumaş"]
    result = []
    parts = str(attr_text).split(",")
    for part in parts:
        for key in important_keys:
            if key in part.lower():
                result.append(part.strip())
                break
    return " ".join(result)[:300]

def build_item_text(row):
    attrs = extract_important_attributes(row["attributes"])
    parts = [
        row["title"],
        row["category"].replace("/", " "),
        row["brand"],
        attrs
    ]
    text = " ".join([p for p in parts if p and str(p).strip() != "" and str(p) != "unknown"])
    return clean_text(text)

items["brand"] = items["brand"].fillna("unknown")
items["attributes"] = items["attributes"].fillna("")
items["item_text"] = items.apply(build_item_text, axis=1)
items["main_category"] = items["category"].str.split("/").str[0]
terms["query"] = terms["query"].apply(clean_text)

print("✅ Temizlik tamam")
print(f"Örnek item_text:\n{items['item_text'].iloc[0][:200]}")

✅ Temizlik tamam
Örnek item_text:
erkek kumaş usb kulaklık çıkışlı bodybag göğüs ve omuz çantası siyah beyaz aksesuar çanta omuz çantası newish polo materyal tekstil renk gri materyal bileşeni astar 100 polyester desen düz kumaş tipi 


## E5 Embedding ile Hard Negative Üretimi

**Amaç:** Modeli kandıracak kadar zor negatif örnekler üretmek.

**Adım 1 — Vektöre Çevirme:**
- 962K ürün ve 17,965 query, E5 modeli ile 768 boyutlu vektörlere dönüştürüldü
- Anlamca benzer metinler matematiksel olarak birbirine yakın vektörler üretir
- "sneaker" ve "koşu ayakkabısı" yakın → "tencere" uzak

**Adım 2 — Hard Negative Bulma:**
- Her query vektörü ile tüm ürün vektörleri karşılaştırıldı
- En yakın 20 ürün seçildi
- Gerçek pozitif olmayanlar ve similarity >= 0.6 olanlar negatif alındı

**Neden E5:**
- Retrieval için özel tasarlanmış, 100 dil biliyor
- Ortalama similarity 0.86 → çok kaliteli zor negatifler

In [4]:
print("E5 modeli yükleniyor...")
e5_model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)
print("✅ E5 hazır")

def get_item_embeddings(texts, batch_size=256):
    prefixed = ["passage: " + t for t in texts]
    return e5_model.encode(prefixed, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

def get_query_embeddings(texts, batch_size=256):
    prefixed = ["query: " + t for t in texts]
    return e5_model.encode(prefixed, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

print("Ürün embeddingleri hesaplanıyor...")
item_texts = items["item_text"].fillna("").tolist()
item_embeddings = get_item_embeddings(item_texts)
np.save(PATH + "item_embeddings.npy", item_embeddings)
print(f"✅ Item embeddings hazır: {item_embeddings.shape}")

train_full = train.merge(terms, on="term_id", how="left")
train_full = train_full.merge(items[["item_id","item_text","main_category"]], on="item_id", how="left")

unique_queries = train_full["query"].unique().tolist()
query_embeddings = get_query_embeddings(unique_queries)
print(f"✅ Query embeddings hazır: {query_embeddings.shape}")

E5 modeli yükleniyor...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ E5 hazır
Ürün embeddingleri hesaplanıyor...


Batches:   0%|          | 0/3762 [00:00<?, ?it/s]

✅ Item embeddings hazır: (962873, 768)


Batches:   0%|          | 0/71 [00:00<?, ?it/s]

✅ Query embeddings hazır: (17965, 768)


In [5]:
import torch.nn.functional as F

print("Hard negativeler bulunuyor...")
item_emb_t = torch.tensor(item_embeddings, device=device, dtype=torch.float32)
query_emb_t = torch.tensor(query_embeddings, device=device, dtype=torch.float32)

item_emb_t = F.normalize(item_emb_t, p=2, dim=1)
query_emb_t = F.normalize(query_emb_t, p=2, dim=1)

item_ids = items["item_id"].tolist()
query_to_pos_items = train_full.groupby("query")["item_id"].apply(set).to_dict()

TOP_K = 20
hard_negatives = []
batch_size_q = 256

for start in tqdm(range(0, len(unique_queries), batch_size_q)):
    end = min(start + batch_size_q, len(unique_queries))
    q_batch = query_emb_t[start:end]
    sims = torch.matmul(q_batch, item_emb_t.T)
    topk_vals, topk_idx = torch.topk(sims, TOP_K, dim=1)
    topk_idx = topk_idx.cpu().numpy()
    topk_vals = topk_vals.cpu().numpy()

    for i, q_idx in enumerate(range(start, end)):
        query_text = unique_queries[q_idx]
        pos_items = query_to_pos_items.get(query_text, set())
        for j, item_idx in enumerate(topk_idx[i]):
            item_id = item_ids[item_idx]
            sim = topk_vals[i][j]
            if item_id not in pos_items and sim >= 0.6:
                hard_negatives.append({
                    "query": query_text,
                    "item_text": item_texts[item_idx],
                    "label": 0,
                    "sim_score": sim
                })

hns_df = pd.DataFrame(hard_negatives)
print(f"✅ E5 hard negative sayısı: {len(hns_df):,}")
print(f"Similarity ortalaması: {hns_df['sim_score'].mean():.4f}")

Hard negativeler bulunuyor...


100%|██████████| 71/71 [01:00<00:00,  1.17it/s]


✅ E5 hard negative sayısı: 323,881
Similarity ortalaması: 0.8558


## Dataset Oluşturma

**Amaç:** Eğitim için dengeli ve kaliteli pozitif + negatif örneklerden oluşan dataset hazırlamak.

**İçerik:**
- 250K pozitif → Trendyol'un verdiği gerçek alakalı çiftler
- 125K orijinal negatif → aynı kategoriden rastgele seçilen alakasız ürünler
- 125K E5 hard negative → anlamca çok yakın ama yanlış ürünler
- Toplam: 500K dengeli dataset (1:1 pozitif/negatif oranı)

**Neden karma negatif:**
- Sadece orijinal negatif → model kolay örnekleri öğrenir, zor örneklerde başarısız olur
- Sadece E5 negatif → model çok seçici olur, gerçek pozitifleri kaçırır  
- Karma → denge sağlanır, Kaggle'da en iyi sonucu bu verdi (0.85)

**Kaydedilen dosyalar:**
- dataset.csv → eğitim verisi
- items_clean.csv → temizlenmiş ürün kataloğu
- terms_clean.csv → temizlenmiş arama terimleri

In [ ]:
np.random.seed(42)

# Pozitif örnekler
pos_df = pd.DataFrame({
    "query": train_full["query"],
    "item_text": train_full["item_text"],
    "label": 1
})

# Orijinal negatif (aynı kategoriden rastgele)
items_clean_neg = items[["item_id","item_text","main_category"]].copy()
neg_items_easy = items_clean_neg.sample(len(train_full), replace=True, random_state=42).reset_index(drop=True)
train_reset = train_full.reset_index(drop=True)
neg_easy = pd.DataFrame({
    "query": train_reset["query"],
    "item_text": neg_items_easy["item_text"].values,
    "label": 0
})
same_cat = neg_items_easy["main_category"].values == train_reset["main_category"].values
neg_easy = neg_easy[~same_cat].reset_index(drop=True)

neg_hard_list = []
for cat in train_full["main_category"].unique():
    cat_trains = train_full[train_full["main_category"] == cat]
    cat_items = items_clean_neg[items_clean_neg["main_category"] == cat]
    if len(cat_items) < 2:
        continue
    neg_sample = cat_items.sample(len(cat_trains), replace=True, random_state=42).reset_index(drop=True)
    cat_trains = cat_trains.reset_index(drop=True)
    mask = neg_sample["item_id"].values != cat_trains["item_id"].values
    neg_hard_list.append(pd.DataFrame({
        "query": cat_trains["query"][mask].values,
        "item_text": neg_sample["item_text"][mask].values,
        "label": 0
    }))
orig_neg = pd.concat(neg_hard_list, ignore_index=True)

# 125K orijinal negatif + 125K E5 negatif
orig_neg_sampled = orig_neg.sample(125000, random_state=42).reset_index(drop=True)
e5_neg_sampled = hns_df[["query","item_text","label"]].sample(125000, random_state=42).reset_index(drop=True)

# Dataset birleştir
dataset = pd.concat([pos_df, orig_neg_sampled, e5_neg_sampled], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Kaydet
dataset.to_csv(PATH + "dataset.csv", index=False)
items[["item_id","item_text","main_category"]].to_csv(PATH + "items_clean.csv", index=False)
terms[["term_id","query"]].to_csv(PATH + "terms_clean.csv", index=False)

print(f"✅ Dataset hazır")
print(f"Toplam  : {len(dataset):,}")
print(f"Pozitif : {(dataset['label']==1).sum():,}")
print(f"Negatif : {(dataset['label']==0).sum():,}")
print("✅ Tüm dosyalar kaydedildi!")

pos_df_orig = pd.DataFrame({"query": train_full["query"], "item_text": train_full["item_text"], "label": 1})
dataset_orig = pd.concat([pos_df_orig, orig_neg_sampled], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
dataset_orig.to_csv(PATH + "dataset_orig.csv", index=False)
print(f"✅ dataset_orig.csv: {len(dataset_orig):,} satır")

✅ Dataset hazır
Toplam  : 500,000
Pozitif : 250,000
Negatif : 250,000
✅ Tüm dosyalar kaydedildi!


✅ dataset_orig.csv: 375,000 satır
